# BirdCLEF2026 - Evaluation and Submission

This notebook handles model evaluation, inference, and submission file generation.

## Why This Step is Critical

1. **Test data is different** - Field recordings from Pantanal
2. **Need proper preprocessing** - Must match training exactly
3. **Submission format** - 12 segments per file, 234 species
4. **No test files locally** - Will be populated on Kaggle

This notebook handles the domain shift challenge!

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Set paths
DATA_DIR = Path("../input")
TEST_SOUNDSCAPES_DIR = DATA_DIR / "test_soundscapes"
TRAIN_SOUNDSCAPES_DIR = DATA_DIR / "train_soundscapes"
OUTPUT_DIR = Path("..")
PREPROCESSED_DATA_DIR = Path('../processed_data')
MODELS_DIR = OUTPUT_DIR / "models"

# Configuration (must match training)
SAMPLE_RATE = 32000
DURATION = 5
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
EXPECTED_SAMPLES = SAMPLE_RATE * DURATION

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Configuration and Models

Load saved configuration and trained model weights.

In [2]:
# Get species columns from saved species mapping
with open(PREPROCESSED_DATA_DIR / 'species_mapping.json', 'r') as f:
    saved_species_to_idx = json.load(f)
species_to_idx = {k: int(v) for k, v in saved_species_to_idx.items()}
idx_to_species = {int(v): k for k, v in saved_species_to_idx.items()}
species_columns = sorted(species_to_idx.keys())
NUM_CLASSES = len(species_columns)

print(f"Number of classes: {NUM_CLASSES}")

Number of classes: 234


In [3]:
import timm
import librosa
import soundfile as sf

class BirdAudioClassifier(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=234, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.num_features = self.backbone.num_features
        self.classifier = nn.Linear(self.num_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.classifier(features)
        return out

model = BirdAudioClassifier('efficientnet_b0', NUM_CLASSES, pretrained=False)
model_path = MODELS_DIR / 'efficientnet_best.pt'
if model_path.exists():
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    print(f"Loaded model from {model_path}")
else:
    print("No trained model found. Using pretrained model without fine-tuning.")
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(device)
    model.eval()

Loaded model from ../models/efficientnet_best.pt


## 2. Audio Processing Functions

Must match training preprocessing exactly.

In [4]:
def load_audio(file_path, target_sr=SAMPLE_RATE):
    try:
        waveform, sr = sf.read(file_path)
        if len(waveform.shape) > 1:
            waveform = waveform.mean(axis=1)
        if sr != target_sr:
            waveform = librosa.resample(waveform, orig_sr=sr, target_sr=target_sr)
        return waveform, target_sr
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

def get_mel_spectrogram(waveform, sr=SAMPLE_RATE):
    mel_spec = librosa.feature.melspectrogram(
        y=waveform, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=50, fmax=15000, power=2.0
    )
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    return log_mel_spec

def normalize_spectrogram(spec):
    spec = spec - spec.min()
    spec = spec / (spec.max() + 1e-8)
    return spec

print("Audio processing functions defined.")

Audio processing functions defined.


## 3. Find Test Soundscape Files

Test files will be populated when running on Kaggle.

In [5]:
test_files = []
for ext in ['*.ogg', '*.mp3', '*.wav']:
    test_files.extend(TEST_SOUNDSCAPES_DIR.glob(ext))
test_files = sorted(set(test_files))
print(f"Found {len(test_files)} test soundscape files")

if len(test_files) == 0:
    print("\n⚠️ No test files found in test_soundscapes directory.")
    print("Note: Test data will be populated when notebook is run on Kaggle.")
    for ext in ['*.ogg', '*.mp3', '*.wav']:
        test_files.extend(TRAIN_SOUNDSCAPES_DIR.glob(ext))
    print(f"Found {len(test_files)} train soundscape files")

Found 0 test soundscape files

⚠️ No test files found in test_soundscapes directory.
Note: Test data will be populated when notebook is run on Kaggle.
Found 10658 train soundscape files


## 4. Generate Submission (Streaming)

Process each test soundscape file one at a time and write one row to submission.csv at a time.

In [6]:
def process_and_write_submission(test_files, device, model, species_columns, output_path):
    """Process each test file one at a time and write rows to submission.csv immediately."""
    first_row = True
    
    with open(output_path, 'w') as f_out:
        for test_file in tqdm(test_files, desc="Processing test soundscapes"):
            waveform, sr = load_audio(test_file)
            if waveform is None:
                continue
            
            if len(waveform.shape) > 1:
                waveform = waveform.mean(axis=1)
            if sr != SAMPLE_RATE:
                waveform = librosa.resample(waveform, orig_sr=sr, target_sr=SAMPLE_RATE)
            
            filename_stem = test_file.stem
            
            for end_time in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60]:
                start_sample = (end_time - 5) * SAMPLE_RATE
                end_sample = end_time * SAMPLE_RATE
                segment = waveform[start_sample:end_sample]
                
                if len(segment) < EXPECTED_SAMPLES:
                    segment = np.pad(segment, (0, EXPECTED_SAMPLES - len(segment)), mode='constant')
                
                spec = get_mel_spectrogram(segment, SAMPLE_RATE)
                spec = normalize_spectrogram(spec)
                spec = torch.FloatTensor(spec).unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)
                
                with torch.no_grad():
                    spec = spec.to(device)
                    output = model(spec)
                    preds = torch.sigmoid(output).cpu().numpy()[0]
                
                row_id = f"{filename_stem}_{end_time}"
                output_row = {'row_id': row_id}
                for i, species in enumerate(species_columns):
                    output_row[species] = preds[i]
                
                if first_row:
                    f_out.write(','.join(output_row.keys()) + '\n')
                    first_row = False
                
                f_out.write(','.join(str(v) for v in output_row.values()) + '\n')
    
    print(f"Submission saved to: {output_path}")

In [7]:
output_path = "submission.csv"

if len(test_files) > 0:
    process_and_write_submission(test_files, device, model, species_columns, output_path)
else:
    print("No test files available to process.")
    print("Creating empty submission file with headers only...")
    with open(output_path, 'w') as f_out:
        f_out.write('row_id,' + ','.join(species_columns) + '\n')

print(f"\nSubmission saved to: {output_path}")

Processing test soundscapes:  19%|███▉                 | 1987/10658 [18:33<1:20:59,  1.78it/s]


KeyboardInterrupt: 

In [8]:
print("Verifying submission...")
submission_check = pd.read_csv(output_path, nrows=10)
print(f"\nSubmission preview (first 10 rows):")
print(submission_check.head())
print(f"\nPrediction statistics (sample):")
for col in species_columns[:5]:
    print(f"  {col}: mean={submission_check[col].mean():.4f}, min={submission_check[col].min():.4f}, max={submission_check[col].max():.4f}")

Verifying submission...

Submission preview (first 10 rows):
                                     row_id   1161364    116570   1176823  \
0   BC2026_Train_1188_S01_20230223_034500_5  0.000567  0.000506  0.000048   
1  BC2026_Train_1188_S01_20230223_034500_10  0.000055  0.000075  0.000005   
2  BC2026_Train_1188_S01_20230223_034500_15  0.000151  0.000083  0.000032   
3  BC2026_Train_1188_S01_20230223_034500_20  0.000008  0.000030  0.000016   
4  BC2026_Train_1188_S01_20230223_034500_25  0.001332  0.000189  0.000069   

    1491113   1595929    209233     22930     22956     22961  ...   whnjay1  \
0  0.000195  0.000083  0.000011  0.000522  0.000302  0.000023  ...  0.000057   
1  0.000404  0.000022  0.000001  0.000155  0.000006  0.000002  ...  0.000018   
2  0.002760  0.000037  0.000004  0.000260  0.000157  0.000010  ...  0.000042   
3  0.000385  0.000019  0.000001  0.000202  0.000030  0.000009  ...  0.000018   
4  0.000482  0.000248  0.000011  0.000175  0.000484  0.000032  ...  0.000086

## Summary

This notebook:

1. **Loaded trained model** - EfficientNet-B0
2. **Processed test soundscapes** - One file at a time
3. **Generated predictions** - For each segment (12 per file), all 234 species
4. **Created submission** - Streaming write, one row at a time
5. **Saved submission.csv** - Ready for Kaggle submission

**Note**: Streaming approach processes one test soundscape at a time and writes one row at a time, avoiding memory issues.